In [ ]:
import pandas as pd
import numpy as np

In [2]:
path = "C:\\Users\\adrme\\OneDrive\\Music\\Project\\Raw_Data"

In [3]:
path1 = r"C:\\Users\\adrme\\OneDrive\\Music\\Project\\Raw_Data\\2023\\12\\01.12.23_NLDC_PSP.xls" 

df = pd.read_excel(
    io=path1, 
    sheet_name="MOP_E", 
    engine='xlrd'
)

print("File successfully loaded into DataFrame (df)!")
print(df.head())

File successfully loaded into DataFrame (df)!
                                          Unnamed: 0 Unnamed: 1 Unnamed: 2  \
0                                                NaN        NaN        NaN   
1                            Report for previous day        NaN        NaN   
2  A. Power Supply Position at All India and Regi...        NaN        NaN   
3                                                NaN        NaN         NR   
4  Demand Met during Evening Peak hrs(MW) (at 19:...        NaN      48007   

  Unnamed: 3 Unnamed: 4 Unnamed: 5          Unnamed: 6 Unnamed: 7   Unnamed: 8  
0        NaN        NaN        NaN                 NaN        NaN          NaN  
1        NaN        NaN        NaN  Date of Reporting:        NaN  02-Dec-2023  
2        NaN        NaN        NaN                 NaN        NaN          NaN  
3         WR         SR         ER                 NER      TOTAL          NaN  
4      51667      46176      19655                2470     167975          NaN  

In [ ]:
import os
from pathlib import Path

path1 = Path(r"C:\Users\adrme\OneDrive\Music\Project")

RAW_DATA_FOLDER = path1 / 'Raw_Data'
CLEANED_DATA1_FOLDER = path1 / 'Cleaned_Data1'
CLEANED_DATA2_FOLDER = path1 / 'Cleaned_Data2'

SOURCE_SHEET_NAME = "MOP_E"

# --- Data Extraction Function ---
def extract_and_save(input_file_path):
    
    # 1. READ BLOCK 1 (Rows 5 to 14) -> 10 rows, skipping the first 4
    df_block1 = pd.read_excel(
        input_file_path,
        sheet_name=SOURCE_SHEET_NAME,
        engine='xlrd',
        header=None,
        skiprows=4,
        nrows=10
    )

    df_block2 = pd.read_excel(
        input_file_path,
        sheet_name=SOURCE_SHEET_NAME,
        engine='xlrd',
        header=None,
        skiprows=18,
        nrows=40
    )
    
    # 3. CONCATENATE BLOCKS (FOR CLEANED_DATA1&2)
  
    month_folder_name = input_file_path.parent.name
    year_folder_name = input_file_path.parent.parent.name
    
    # Construct output directory paths
    output_dir_1 = CLEANED_DATA1_FOLDER / year_folder_name / month_folder_name
    output_dir_2 = CLEANED_DATA2_FOLDER / year_folder_name / month_folder_name
    
    # Create nested directories safely (exist_ok=True prevents error if folders exist)
    os.makedirs(output_dir_1, exist_ok=True)
    os.makedirs(output_dir_2, exist_ok=True)
    
  
    output_file_name = input_file_path.stem + '.xlsx'
    
    # Save to Cleaned_Data1 (Rows 5-14)
    output_path_1 = output_dir_1 / output_file_name
    df_block1.to_excel(output_path_1, index=False, header=False)
    
    # Save to Cleaned_Data2 (Rows 19-58)
    output_path_2 = output_dir_2 / output_file_name
    df_block2.to_excel(output_path_2, index=False, header=False)
    
    print(f"Processed: {input_file_path.name} -> Saved to two locations.")


# --- Main Execution ---

# Use rglob to recursively find all .xls files
xls_files = list(RAW_DATA_FOLDER.rglob('*.xls'))

if not xls_files:
    print("No .xls files found in the Raw_Data directory structure.")
else:
    print(f"Found {len(xls_files)} files to process.")
    
    for input_file_path in xls_files:
        try:
            extract_and_save(input_file_path)

        except Exception as e:
            print(f"Error processing file {input_file_path.name}: {e}")

print("\nProcessing complete.")

Found 929 files to process.
Processed: 01.10.23_NLDC_PSP.xls -> Saved to two locations.
Processed: 02.10.23_NLDC_PSP.xls -> Saved to two locations.
Processed: 03.10.23_NLDC_PSP.xls -> Saved to two locations.
Processed: 04.10.23_NLDC_PSP.xls -> Saved to two locations.
Processed: 05.10.23_NLDC_PSP.xls -> Saved to two locations.
Processed: 06.10.23_NLDC_PSP.xls -> Saved to two locations.
Processed: 07.10.23_NLDC_PSP.xls -> Saved to two locations.
Processed: 08.10.23_NLDC_PSP.xls -> Saved to two locations.
Processed: 09.10.23_NLDC_PSP.xls -> Saved to two locations.
Processed: 10.10.23_NLDC_PSP.xls -> Saved to two locations.
Processed: 11.10.23_NLDC_PSP.xls -> Saved to two locations.
Processed: 12.10.23_NLDC_PSP.xls -> Saved to two locations.
Processed: 13.10.23_NLDC_PSP.xls -> Saved to two locations.
Processed: 14.10.23_NLDC_PSP.xls -> Saved to two locations.
Processed: 15.10.23_NLDC_PSP.xls -> Saved to two locations.
Processed: 16.10.23_NLDC_PSP.xls -> Saved to two locations.
Processed: 1

In [ ]:
import pandas as pd
from pathlib import Path

# --- Configuration ---
path1 = Path(r"C:\Users\adrme\OneDrive\Music\Project")
CLEANED_DATA1_FOLDER = path1 / 'Cleaned_Data1'

# --- Main Modification Logic ---
def delete_header_row_cleaned1():
    
    # Recursively find all .xlsx files ONLY in the Cleaned_Data1 folder using rglob
    xlsx_files = list(CLEANED_DATA1_FOLDER.rglob('*.xlsx'))

    if not xlsx_files:
        print("No .xlsx files found in the Cleaned_Data1 directory structure.")
        return
        
    print(f"Found {len(xlsx_files)} files in Cleaned_Data1 to modify.")

    for file_path in xlsx_files:
        try:
            # 1. Load the existing data. header=0 reads the current header row (index 0).
            df = pd.read_excel(file_path, header=0)
            
            # 2. Delete the header row. The current header (the column names) is now gone.
            # We must load the data again without a header, then drop the first data row.
            
            # Simpler method: Load the file again, skipping the first row (the header)
            df_no_header = pd.read_excel(file_path, header=None, skiprows=1)
            
            # 3. Overwrite the existing file with the data-only DataFrame.
            # header=False ensures no new header is written.
            df_no_header.to_excel(file_path, index=False, header=False)
            
            print(f"Modified: Header deleted from {file_path.name}")

        except Exception as e:
            print(f"Error modifying file {file_path.name}: {e}")

    print("\nModification complete for all files in Cleaned_Data1.")

delete_header_row_cleaned1()

Found 929 files in Cleaned_Data1 to modify.
Modified: Header deleted from 01.10.23_NLDC_PSP.xlsx
Modified: Header deleted from 02.10.23_NLDC_PSP.xlsx
Modified: Header deleted from 03.10.23_NLDC_PSP.xlsx
Modified: Header deleted from 04.10.23_NLDC_PSP.xlsx
Modified: Header deleted from 05.10.23_NLDC_PSP.xlsx
Modified: Header deleted from 06.10.23_NLDC_PSP.xlsx
Modified: Header deleted from 07.10.23_NLDC_PSP.xlsx
Modified: Header deleted from 08.10.23_NLDC_PSP.xlsx
Modified: Header deleted from 09.10.23_NLDC_PSP.xlsx
Modified: Header deleted from 10.10.23_NLDC_PSP.xlsx
Modified: Header deleted from 11.10.23_NLDC_PSP.xlsx
Modified: Header deleted from 12.10.23_NLDC_PSP.xlsx
Modified: Header deleted from 13.10.23_NLDC_PSP.xlsx
Modified: Header deleted from 14.10.23_NLDC_PSP.xlsx
Modified: Header deleted from 15.10.23_NLDC_PSP.xlsx
Modified: Header deleted from 16.10.23_NLDC_PSP.xlsx
Modified: Header deleted from 17.10.23_NLDC_PSP.xlsx
Modified: Header deleted from 18.10.23_NLDC_PSP.xlsx
Mo

In [ ]:
import pandas as pd
from pathlib import Path

# --- Configuration ---
path1 = Path(r"C:\Users\adrme\OneDrive\Music\Project")
CLEANED_DATA1_FOLDER = path1 / 'Cleaned_Data1'

# Indices of the rows to delete (0-based):
# (Row 4, Row 5, Row 6, Row 9, Row 10 from the visible data block)
ROWS_TO_DELETE = [3, 4, 5, 8, 9]

# --- Main Modification Logic ---
def delete_specific_rows_cleaned1():
    
    # Recursively find all .xlsx files ONLY in the Cleaned_Data1 folder
    xlsx_files = list(CLEANED_DATA1_FOLDER.rglob('*.xlsx'))

    if not xlsx_files:
        print("No .xlsx files found in the Cleaned_Data1 directory structure.")
        return
        
    print(f"Found {len(xlsx_files)} files in Cleaned_Data1 to modify.")

    for file_path in xlsx_files:
        try:
            # 1. Load the existing data. header=None since headers were deleted previously.
            df = pd.read_excel(file_path, header=None)
            
            # 2. Delete the specified rows by index
            # axis=0 means delete rows
            df.drop(index=ROWS_TO_DELETE, axis=0, inplace=True, errors='ignore')
            
            # 3. Reset the index of the remaining rows (optional, but good practice)
            df.reset_index(drop=True, inplace=True)
            
            # 4. Overwrite the existing file. header=False as requested.
            df.to_excel(file_path, index=False, header=False)
            
            print(f"Modified: Deleted specified rows from {file_path.name}")

        except Exception as e:
            print(f"Error modifying file {file_path.name}: {e}")

    print("\nModification complete for all files in Cleaned_Data1.")

# Execute the function
delete_specific_rows_cleaned1()

Found 929 files in Cleaned_Data1 to modify.
Modified: Deleted specified rows from 01.10.23_NLDC_PSP.xlsx
Modified: Deleted specified rows from 02.10.23_NLDC_PSP.xlsx
Modified: Deleted specified rows from 03.10.23_NLDC_PSP.xlsx
Modified: Deleted specified rows from 04.10.23_NLDC_PSP.xlsx
Modified: Deleted specified rows from 05.10.23_NLDC_PSP.xlsx
Modified: Deleted specified rows from 06.10.23_NLDC_PSP.xlsx
Modified: Deleted specified rows from 07.10.23_NLDC_PSP.xlsx
Modified: Deleted specified rows from 08.10.23_NLDC_PSP.xlsx
Modified: Deleted specified rows from 09.10.23_NLDC_PSP.xlsx
Modified: Deleted specified rows from 10.10.23_NLDC_PSP.xlsx
Modified: Deleted specified rows from 11.10.23_NLDC_PSP.xlsx
Modified: Deleted specified rows from 12.10.23_NLDC_PSP.xlsx
Modified: Deleted specified rows from 13.10.23_NLDC_PSP.xlsx
Modified: Deleted specified rows from 14.10.23_NLDC_PSP.xlsx
Modified: Deleted specified rows from 15.10.23_NLDC_PSP.xlsx
Modified: Deleted specified rows from 16.

In [ ]:
import pandas as pd
from pathlib import Path

# --- Configuration ---
path1 = Path(r"C:\Users\adrme\OneDrive\Music\Project")
CLEANED_DATA1_FOLDER = path1 / 'Cleaned_Data1'

# --- New Header Definition (8 Columns) ---
NEW_HEADERS_C1 = [
    'Parameter',
    'extra_1',  # Renaming 'extra' columns for uniqueness
    'extra_2',
    'extra_3',
    'extra_4',
    'extra_5',
    'extra_6',
    'Total'
]

# --- Main Modification Logic ---
def add_headers_cleaned1():
    
    # Recursively find all .xlsx files in the Cleaned_Data1 folder
    xlsx_files = list(CLEANED_DATA1_FOLDER.rglob('*.xlsx'))

    if not xlsx_files:
        print("No .xlsx files found in the Cleaned_Data1 directory structure.")
        return
        
    print(f"Found {len(xlsx_files)} files in Cleaned_Data1 to modify.")

    for file_path in xlsx_files:
        try:
            # 1. Load the existing data. header=None because headers were deleted previously.
            df = pd.read_excel(file_path, header=None)
            
            # 2. Check for the required 8 columns (1 label + 7 value columns)
            if df.shape[1] != len(NEW_HEADERS_C1):
                 print(f"Column count mismatch in {file_path.name}: Found {df.shape[1]} columns, expected {len(NEW_HEADERS_C1)}. Skipping file.")
                 continue

            # 3. Apply the new header list.
            df.columns = NEW_HEADERS_C1
            
            # 4. Overwrite the existing file. header=True writes the new column names.
            df.to_excel(file_path, index=False, header=True)
            
            print(f"Modified: Added new headers to {file_path.name}")

        except Exception as e:
            print(f"Error modifying file {file_path.name}: {e}")

    print("\nModification complete for all files in Cleaned_Data1.")

# Execute the function
add_headers_cleaned1()

Found 929 files in Cleaned_Data1 to modify.
Modified: Added new headers to 01.10.23_NLDC_PSP.xlsx
Modified: Added new headers to 02.10.23_NLDC_PSP.xlsx
Modified: Added new headers to 03.10.23_NLDC_PSP.xlsx
Modified: Added new headers to 04.10.23_NLDC_PSP.xlsx
Modified: Added new headers to 05.10.23_NLDC_PSP.xlsx
Modified: Added new headers to 06.10.23_NLDC_PSP.xlsx
Modified: Added new headers to 07.10.23_NLDC_PSP.xlsx
Modified: Added new headers to 08.10.23_NLDC_PSP.xlsx
Modified: Added new headers to 09.10.23_NLDC_PSP.xlsx
Modified: Added new headers to 10.10.23_NLDC_PSP.xlsx
Modified: Added new headers to 11.10.23_NLDC_PSP.xlsx
Modified: Added new headers to 12.10.23_NLDC_PSP.xlsx
Modified: Added new headers to 13.10.23_NLDC_PSP.xlsx
Modified: Added new headers to 14.10.23_NLDC_PSP.xlsx
Modified: Added new headers to 15.10.23_NLDC_PSP.xlsx
Modified: Added new headers to 16.10.23_NLDC_PSP.xlsx
Modified: Added new headers to 17.10.23_NLDC_PSP.xlsx
Modified: Added new headers to 18.10.2

In [ ]:
import pandas as pd
from pathlib import Path

# --- Configuration ---
path1 = Path(r"C:\Users\adrme\OneDrive\Music\Project")
CLEANED_DATA1_FOLDER = path1 / 'Cleaned_Data1'

# --- Main Modification Logic ---
def drop_extra_columns_c1():
    
    # Recursively find all .xlsx files in the Cleaned_Data1 folder
    xlsx_files = list(CLEANED_DATA1_FOLDER.rglob('*.xlsx'))

    if not xlsx_files:
        print("No .xlsx files found in the Cleaned_Data1 directory structure.")
        return
        
    print(f"Found {len(xlsx_files)} files in Cleaned_Data1 to modify.")

    for file_path in xlsx_files:
        try:
            # 1. Load the existing data. header=0 ensures the current header row is read into df.
            df = pd.read_excel(file_path, header=0)
            
            # 2. Identify columns to drop: any column name containing "extra" (case-insensitive)
            # This handles 'extra_1', 'extra_2', etc., but keeps 'Parameter' and 'Total'.
            columns_to_drop = [col for col in df.columns if 'extra' in col.lower()]
            
            # 3. Drop the identified columns
            if columns_to_drop:
                # errors='ignore' ensures the loop doesn't fail if a column is missing
                df.drop(columns=columns_to_drop, axis=1, inplace=True, errors='ignore')
            
            # 4. Overwrite the existing file with the modified DataFrame.
            df.to_excel(file_path, index=False, header=True)
            
            print(f"Modified: Dropped {len(columns_to_drop)} 'extra' column(s) from {file_path.name}")

        except Exception as e:
            print(f"Error modifying file {file_path.name}: {e}")

    print("\nModification complete for all files in Cleaned_Data1.")

# Execute the function
drop_extra_columns_c1()

Found 929 files in Cleaned_Data1 to modify.
Modified: Dropped 6 'extra' column(s) from 01.10.23_NLDC_PSP.xlsx
Modified: Dropped 6 'extra' column(s) from 02.10.23_NLDC_PSP.xlsx
Modified: Dropped 6 'extra' column(s) from 03.10.23_NLDC_PSP.xlsx
Modified: Dropped 6 'extra' column(s) from 04.10.23_NLDC_PSP.xlsx
Modified: Dropped 6 'extra' column(s) from 05.10.23_NLDC_PSP.xlsx
Modified: Dropped 6 'extra' column(s) from 06.10.23_NLDC_PSP.xlsx
Modified: Dropped 6 'extra' column(s) from 07.10.23_NLDC_PSP.xlsx
Modified: Dropped 6 'extra' column(s) from 08.10.23_NLDC_PSP.xlsx
Modified: Dropped 6 'extra' column(s) from 09.10.23_NLDC_PSP.xlsx
Modified: Dropped 6 'extra' column(s) from 10.10.23_NLDC_PSP.xlsx
Modified: Dropped 6 'extra' column(s) from 11.10.23_NLDC_PSP.xlsx
Modified: Dropped 6 'extra' column(s) from 12.10.23_NLDC_PSP.xlsx
Modified: Dropped 6 'extra' column(s) from 13.10.23_NLDC_PSP.xlsx
Modified: Dropped 6 'extra' column(s) from 14.10.23_NLDC_PSP.xlsx
Modified: Dropped 6 'extra' colu

In [ ]:
import pandas as pd
from pathlib import Path

# --- Configuration ---
path1 = Path(r"C:\Users\adrme\OneDrive\Music\Project")
CLEANED_DATA2_FOLDER = path1 / 'Cleaned_Data2'

# --- Main Modification Logic ---
def delete_column_a_cleaned2():
    
    # Recursively find all .xlsx files in the Cleaned_Data2 folder
    xlsx_files = list(CLEANED_DATA2_FOLDER.rglob('*.xlsx'))

    if not xlsx_files:
        print("No .xlsx files found in the Cleaned_Data2 directory structure.")
        return
        
    print(f"Found {len(xlsx_files)} files in Cleaned_Data2 to modify.")

    for file_path in xlsx_files:
        try:
            # 1. Load the existing data. header=0 assumes the first row is the header.
            # If the files are corrupted, we assume the first column is always the one to delete.
            df = pd.read_excel(file_path, header=0)
            
            # 2. Delete the column at index 0 (Column A).
            df.drop(columns=df.columns[0], axis=1, inplace=True)
            
            # 3. Overwrite the existing file with the modified DataFrame.
            df.to_excel(file_path, index=False, header=True)
            
            print(f"Modified: Deleted Column A (Region) from {file_path.name}")

        except Exception as e:
            print(f"Error modifying file {file_path.name}: {e}")

    print("\nModification complete for all files in Cleaned_Data2.")

# Execute the function
delete_column_a_cleaned2()

Found 930 files in Cleaned_Data2 to modify.
Modified: Deleted Column A (Region) from 01.10.23_NLDC_PSP.xlsx
Modified: Deleted Column A (Region) from 02.10.23_NLDC_PSP.xlsx
Modified: Deleted Column A (Region) from 03.10.23_NLDC_PSP.xlsx
Modified: Deleted Column A (Region) from 04.10.23_NLDC_PSP.xlsx
Modified: Deleted Column A (Region) from 05.10.23_NLDC_PSP.xlsx
Modified: Deleted Column A (Region) from 06.10.23_NLDC_PSP.xlsx
Modified: Deleted Column A (Region) from 07.10.23_NLDC_PSP.xlsx
Modified: Deleted Column A (Region) from 08.10.23_NLDC_PSP.xlsx
Modified: Deleted Column A (Region) from 09.10.23_NLDC_PSP.xlsx
Modified: Deleted Column A (Region) from 10.10.23_NLDC_PSP.xlsx
Modified: Deleted Column A (Region) from 11.10.23_NLDC_PSP.xlsx
Modified: Deleted Column A (Region) from 12.10.23_NLDC_PSP.xlsx
Modified: Deleted Column A (Region) from 13.10.23_NLDC_PSP.xlsx
Modified: Deleted Column A (Region) from 14.10.23_NLDC_PSP.xlsx
Modified: Deleted Column A (Region) from 15.10.23_NLDC_PSP.x

In [ ]:
import pandas as pd
from pathlib import Path

# --- Configuration ---
path1 = Path(r"C:\Users\adrme\OneDrive\Music\Project")
CLEANED_DATA2_FOLDER = path1 / 'Cleaned_Data2'

# --- New Header Definition (8 Columns) ---
NEW_HEADERS_C2 = [
    'States',
    'Max Demand Met(MW)',
    'extra_1',
    'Energy Met(MU)',
    'extra_2',
    'extra_3',
    'extra_4',
    'Energy Shortage(MU)'
]
REQUIRED_COLUMNS = len(NEW_HEADERS_C2)

# --- Main Modification Logic ---
def slice_data_and_replace_header_c2():
    
    xlsx_files = list(CLEANED_DATA2_FOLDER.rglob('*.xlsx'))

    if not xlsx_files:
        print("No .xlsx files found in the Cleaned_Data2 directory structure.")
        return
        
    print(f"Found {len(xlsx_files)} files in Cleaned_Data2 to modify.")

    for file_path in xlsx_files:
        try:
            # 1. Load the data using the default header (first row is header).
            # This is done to preserve the column count and structure.
            df = pd.read_excel(file_path, header=0) 
            
            # 2. Slice the data rows (Row 3 to Row 40).
            # Since the header is row 1 (index 0), Row 3 starts at index 2, and Row 40 is index 39.
            # df.iloc[2:40] selects rows with indices 2 through 39 (38 rows total).
            df_sliced = df.iloc[2:40].copy()
            
            # 3. Ensure the DataFrame has exactly 8 columns (REQUIRED_COLUMNS)
            # This step attempts to handle any previous column corruption by trimming/padding the columns.
            if df_sliced.shape[1] > REQUIRED_COLUMNS:
                # Trim extra columns if too wide
                df_sliced = df_sliced.iloc[:, :REQUIRED_COLUMNS].copy()
            elif df_sliced.shape[1] < REQUIRED_COLUMNS:
                # Add missing columns if too narrow
                for _ in range(REQUIRED_COLUMNS - df_sliced.shape[1]):
                    df_sliced[df_sliced.shape[1]] = pd.NA
            
            # 4. Apply the new 8-column header list.
            df_sliced.columns = NEW_HEADERS_C2
            
            # 5. Overwrite the existing file.
            df_sliced.to_excel(file_path, index=False, header=True)
            
            print(f"Modified: Sliced rows 3-40 and replaced header in {file_path.name}")

        except Exception as e:
            print(f"Error modifying file {file_path.name}: {e}")

    print("\nModification complete for all files in Cleaned_Data2.")

# Execute the function
slice_data_and_replace_header_c2()

Found 930 files in Cleaned_Data2 to modify.
Modified: Sliced rows 3-40 and replaced header in 01.10.23_NLDC_PSP.xlsx
Modified: Sliced rows 3-40 and replaced header in 02.10.23_NLDC_PSP.xlsx
Modified: Sliced rows 3-40 and replaced header in 03.10.23_NLDC_PSP.xlsx
Modified: Sliced rows 3-40 and replaced header in 04.10.23_NLDC_PSP.xlsx
Modified: Sliced rows 3-40 and replaced header in 05.10.23_NLDC_PSP.xlsx
Modified: Sliced rows 3-40 and replaced header in 06.10.23_NLDC_PSP.xlsx
Modified: Sliced rows 3-40 and replaced header in 07.10.23_NLDC_PSP.xlsx
Modified: Sliced rows 3-40 and replaced header in 08.10.23_NLDC_PSP.xlsx
Modified: Sliced rows 3-40 and replaced header in 09.10.23_NLDC_PSP.xlsx
Modified: Sliced rows 3-40 and replaced header in 10.10.23_NLDC_PSP.xlsx
Modified: Sliced rows 3-40 and replaced header in 11.10.23_NLDC_PSP.xlsx
Modified: Sliced rows 3-40 and replaced header in 12.10.23_NLDC_PSP.xlsx
Modified: Sliced rows 3-40 and replaced header in 13.10.23_NLDC_PSP.xlsx
Modifie

In [5]:
import pandas as pd
from pathlib import Path

# --- Configuration ---
path1 = Path(r"C:\Users\adrme\OneDrive\Music\Project")
CLEANED_DATA2_FOLDER = path1 / 'Cleaned_Data2'

# --- Main Modification Logic ---
def drop_extra_columns_c2_final():
    
    # Recursively find all .xlsx files in the Cleaned_Data2 folder
    xlsx_files = list(CLEANED_DATA2_FOLDER.rglob('*.xlsx'))

    if not xlsx_files:
        print("No .xlsx files found in the Cleaned_Data2 directory structure. Please ensure previous steps ran correctly.")
        return
        
    print(f"Found {len(xlsx_files)} files in Cleaned_Data2 to modify.")

    for file_path in xlsx_files:
        try:
            # Load the existing data. header=0 ensures the current header row is read into df.
            df = pd.read_excel(file_path, header=0)
            
            # Identify columns to drop: any column name containing "extra" (case-insensitive)
            columns_to_drop = [col for col in df.columns if 'extra' in col.lower()]
            
            # Drop the identified columns
            if columns_to_drop:
                # errors='ignore' ensures the loop doesn't fail if a column is missing
                df.drop(columns=columns_to_drop, axis=1, inplace=True, errors='ignore')
            
            # Overwrite the existing file with the modified DataFrame.
            df.to_excel(file_path, index=False, header=True)
            
            print(f"Modified: Dropped {len(columns_to_drop)} 'extra' column(s) from {file_path.name}")

        except Exception as e:
            print(f"Error modifying file {file_path.name}: {e}")

    print("\nModification complete for all files in Cleaned_Data2.")

# Execute the function
drop_extra_columns_c2_final()

Found 930 files in Cleaned_Data2 to modify.
Modified: Dropped 4 'extra' column(s) from 01.10.23_NLDC_PSP.xlsx
Modified: Dropped 4 'extra' column(s) from 02.10.23_NLDC_PSP.xlsx
Modified: Dropped 4 'extra' column(s) from 03.10.23_NLDC_PSP.xlsx
Modified: Dropped 4 'extra' column(s) from 04.10.23_NLDC_PSP.xlsx
Modified: Dropped 4 'extra' column(s) from 05.10.23_NLDC_PSP.xlsx
Modified: Dropped 4 'extra' column(s) from 06.10.23_NLDC_PSP.xlsx
Modified: Dropped 4 'extra' column(s) from 07.10.23_NLDC_PSP.xlsx
Modified: Dropped 4 'extra' column(s) from 08.10.23_NLDC_PSP.xlsx
Modified: Dropped 4 'extra' column(s) from 09.10.23_NLDC_PSP.xlsx
Modified: Dropped 4 'extra' column(s) from 10.10.23_NLDC_PSP.xlsx
Modified: Dropped 4 'extra' column(s) from 11.10.23_NLDC_PSP.xlsx
Modified: Dropped 4 'extra' column(s) from 12.10.23_NLDC_PSP.xlsx
Modified: Dropped 4 'extra' column(s) from 13.10.23_NLDC_PSP.xlsx
Modified: Dropped 4 'extra' column(s) from 14.10.23_NLDC_PSP.xlsx
Modified: Dropped 4 'extra' colu

In [ ]:
import pandas as pd
from pathlib import Path

# --- Configuration ---
path1 = Path(r"C:\Users\adrme\OneDrive\Music\Project")
CLEANED_DATA2_FOLDER = path1 / 'Cleaned_Data2'

# --- Final Output Headers ---
FINAL_HEADERS_TO_ADD = [
    'Day Demand (MU)',
    'Percentage Met',
    'Load Factor'
]

# --- Main Processing Function ---
def calculate_final_features_by_name_c2():
    
    xlsx_files = list(CLEANED_DATA2_FOLDER.rglob('*.xlsx'))

    if not xlsx_files:
        print("No .xlsx files found in the Cleaned_Data2 directory structure.")
        return
        
    print(f"Found {len(xlsx_files)} files to finalize feature calculation.")

    # Define Column Names for Calculation (Based on the structure you confirmed)
    COL_STATES = 'States'
    COL_MAX_DEMAND_MET = 'Max Demand Met(MW)'
    COL_ENERGY_MET = 'Energy Met(MU)'
    COL_ENERGY_SHORTAGE = 'Energy Shortage(MU)'

    for file_path in xlsx_files:
        try:
            # 1. Load the existing data.
            df = pd.read_excel(file_path, header=0)
            
            # --- Feature Calculation (Adding new columns) ---
            
            # 2. Calculate Day Demand (DD) in MU (ADD)
            df['Day Demand (MU)'] = df[COL_ENERGY_MET] + df[COL_ENERGY_SHORTAGE]
            
            # 3. Calculate Percentage Met (P) (%) (ADD)
            df['Percentage Met'] = (
                (df[COL_ENERGY_MET] / df['Day Demand (MU)']) * 100
            ).where(df['Day Demand (MU)'] != 0, 100)
            
            # 4. Calculate Load Factor (LF) (ADD)
            # Average Demand (MW) = (Energy Met (MU) * 1000) / 24
            df['Load Factor'] = (
                (df[COL_ENERGY_MET] * 1000) / 24
            ) / df[COL_MAX_DEMAND_MET].where(df[COL_MAX_DEMAND_MET] != 0, 1)

            # 5. Overwrite the existing file with the DataFrame containing all original columns + 3 new columns.
            # No column selection is needed here; the new columns are simply added to the end.
            df.to_excel(file_path, index=False, header=True)
            
            print(f"Added 3 feature columns to {file_path.name}")

        except KeyError as ke:
            # Catches the error if a column name is inconsistent across files
            print(f"❌ ERROR: Missing required column for calculation in {file_path.name}. Cannot find column: {ke}. Skipping file.")
        except Exception as e:
            print(f"Error processing file {file_path.name}: {e}")

    print("\nFeature calculation complete for Cleaned_Data2. Data was added, not replaced.")

# Execute the function
calculate_final_features_by_name_c2()

Found 930 files to finalize feature calculation.
Added 3 feature columns to 01.10.23_NLDC_PSP.xlsx
Added 3 feature columns to 02.10.23_NLDC_PSP.xlsx
Added 3 feature columns to 03.10.23_NLDC_PSP.xlsx
Added 3 feature columns to 04.10.23_NLDC_PSP.xlsx
Added 3 feature columns to 05.10.23_NLDC_PSP.xlsx
Added 3 feature columns to 06.10.23_NLDC_PSP.xlsx
Added 3 feature columns to 07.10.23_NLDC_PSP.xlsx
Added 3 feature columns to 08.10.23_NLDC_PSP.xlsx
Added 3 feature columns to 09.10.23_NLDC_PSP.xlsx
Added 3 feature columns to 10.10.23_NLDC_PSP.xlsx
Added 3 feature columns to 11.10.23_NLDC_PSP.xlsx
Added 3 feature columns to 12.10.23_NLDC_PSP.xlsx
Added 3 feature columns to 13.10.23_NLDC_PSP.xlsx
Added 3 feature columns to 14.10.23_NLDC_PSP.xlsx
Added 3 feature columns to 15.10.23_NLDC_PSP.xlsx
Added 3 feature columns to 16.10.23_NLDC_PSP.xlsx
Added 3 feature columns to 17.10.23_NLDC_PSP.xlsx
Added 3 feature columns to 18.10.23_NLDC_PSP.xlsx
Added 3 feature columns to 19.10.23_NLDC_PSP.xlsx
A